# 모듈 (4) NVIDIA NeMo-Agent-Toolkit (NAT) 실습

## 에이전트 거버넌스 (Governance)

---

### 학습 목표

1. **NeMo-Agent-Toolkit(NAT)** 의 목적과 위치 이해 — 프레임워크를 대체하지 않고 **감싸서** 관측·평가·최적화
2. 아키텍처와 핵심 기능(Building / Insights / Optimization / Protocol) 파악
3. **NAT 전용 venv 를 만들고 `nvidia-nat` 을 설치**한 뒤, `workflow.yml` 한 장으로 워크플로를 선언하고
   **`nat run` 으로 실제 실행**해 본다
4. `nat info components` 로 NAT 에 **등록된 컴포넌트(= 각 블록의 `_type` 후보)** 를 조회해 플러그인 구조를 확인한다
5. `logging_middleware` 로 **LLM 에 실제로 전송된 메시지와 응답·토큰 사용량**을 관측한다
6. 같은 방식으로 **검색 도구를 쓰는 에이전트 워크플로**를 관측해, 도구 호출이 LLM 왕복·토큰을 어떻게 늘리는지 본다

> 📦 **환경 설치 안내**는 [`env_guides/M03_4_nemo_agent_toolkit.md`](env_guides/M03_4_nemo_agent_toolkit.md) 에 정리되어 있습니다.

### 전제조건

- Windows 11 + **CMD(`cmd.exe`)** + Python **3.11**(`uv` 관리), 커널 = **`Agentic AI (uv)`**
- LLM 은 `notebooks/.env` 의 **`LLM_PROVIDER`** 를 그대로 사용합니다(코드에서 공급자를 강제하지 않음)
- 인터넷 연결(최초 1회 `nvidia-nat` 설치, 약 수백 MB · 6-3절의 위키백과 검색)

### 이 노트북의 실행 방식 — 두 환경을 오가며 진행합니다

NAT 는 LangChain 등 공통 의존성을 **자체 범위로 고정**하므로, 강의 환경(`.venv`)에 직접 설치하면
다른 노트북의 패키지 버전이 흔들릴 수 있습니다. 그래서 **NeMo Guardrails 를 Docker 로 분리한 것과 같은 취지**로
NAT 는 **전용 venv(`notebooks/NAT_Tutorial/.venv-nat`)** 에 두고, 이 노트북은 강의 커널에서
`utils.run_cmd()` 로 그 venv 의 `nat` CLI 를 호출합니다.

```
[ 이 노트북 · 강의 커널 (Agentic AI (uv), Python 3.11) ]
        │  utils.run_cmd("...\NAT_Tutorial\.venv-nat\Scripts\nat.exe run ...")
        ▼
[ NAT 전용 venv (Python 3.11) · nvidia-nat ]  ──→  [ .env 의 LLM_PROVIDER 엔드포인트 ]
```

> 참고: Python 3.11 로 올라오면서 NAT 의 **버전 요구(`>=3.11,<3.13`)는 강의 환경과 이미 호환**됩니다.
> 전용 venv 를 쓰는 이유는 이제 버전이 아니라 **의존성 격리**입니다.

---

In [1]:
# [setup] 자기완결적 셋업 — 이 노트북만 단독으로 실행 가능하게 한다
import os
import sys

sys.path.insert(0, os.path.abspath(''))   # notebooks/ 를 import 경로에 추가

import utils
utils.reload_env()                        # .env 재로드 (LLM_PROVIDER 등 갱신) + 공급자 상태 출력

# 경로 상수 — 이 노트북은 강의 커널에서 'NAT 전용 venv' 의 CLI 를 호출한다
NOTEBOOKS_DIR = os.path.abspath('')                          # ...\notebooks
PROJECT_ROOT  = os.path.dirname(NOTEBOOKS_DIR)               # ...\Agentic AI Tutorial
NAT_DIR       = os.path.join(NOTEBOOKS_DIR, 'NAT_Tutorial')  # NAT 실습 시리즈 폴더
NAT_VENV      = os.path.join(NAT_DIR, '.venv-nat')
NAT_PY        = os.path.join(NAT_VENV, 'Scripts', 'python.exe')
NAT_EXE       = os.path.join(NAT_VENV, 'Scripts', 'nat.exe')
NAT_CONFIG_DIR = os.path.join(NOTEBOOKS_DIR, 'nat_config')   # 이 노트북이 만드는 workflow.yml 위치
WORKFLOW_YML   = os.path.join(NAT_CONFIG_DIR, 'workflow.yml')
os.makedirs(NAT_CONFIG_DIR, exist_ok=True)

# NAT CLI 는 Windows 기본 코드페이지(cp949)에서 한글 출력이 깨질 수 있어 UTF-8 을 강제한다.
# (자식 프로세스가 상속하므로 run_cmd 로 실행되는 nat CLI 에 그대로 적용된다)
os.environ['PYTHONUTF8'] = '1'

print(f"현재 커널 Python : {sys.version.split()[0]}")
print(f"NAT 전용 venv    : {NAT_VENV}")
print(f"workflow.yml     : {WORKFLOW_YML}")

LLM 공급자: nvidia
  NVIDIA build Key: 설정됨  /  Model: meta/llama-3.1-8b-instruct
현재 커널 Python : 3.11.15
NAT 전용 venv    : c:\Users\stshin\Documents\GitHub\Agentic AI Tutorial\notebooks\NAT_Tutorial\.venv-nat
workflow.yml     : c:\Users\stshin\Documents\GitHub\Agentic AI Tutorial\notebooks\nat_config\workflow.yml


## 1. NeMo-Agent-Toolkit 이란?

**NeMo-Agent-Toolkit(NAT)** 은 NVIDIA 가 공개한 **오픈소스 라이브러리**로, 여러 프레임워크로 만든
**AI 에이전트 팀을 효율적으로 연결·최적화**하는 것을 목표로 합니다. 기존 에이전트 프레임워크를
**대체하지 않고 감싸(wrapping)**, 엔터프라이즈급 계측(instrumentation)·관측(observability)·지속 개선을 더합니다.

> 이 프로젝트는 이전에 **Agent Intelligence Toolkit (AIQ toolkit)** 라는 이름으로 알려졌습니다(리브랜딩).

- 즉, **프레임워크 무관(framework-agnostic)** 계층입니다. LangChain·LlamaIndex·CrewAI 등으로 만든 에이전트를
  그대로 두고, 그 위에서 **프로파일링 · 평가 · 관측**을 수행합니다.
- 이는 본 모듈의 주제인 **거버넌스(통제·관측)** 와 직접 맞닿아 있습니다 — 가드레일(안전)·트레이싱(관측)의
  연장선에서, 프레임워크에 얽매이지 않는 **표준 관측/평가 도구**를 제공합니다.

### 지원 프레임워크

LangChain, LlamaIndex, CrewAI, Microsoft Semantic Kernel, Google ADK, 그리고 커스텀 Python 에이전트.

### 프로토콜 지원

**MCP(Model Context Protocol)** 와 **A2A(Agent-to-Agent)** 프로토콜 연동 — 본 강의 모듈 1에서 다룬 개념과 연결됩니다.

> 출처: NVIDIA/NeMo-Agent-Toolkit GitHub — https://github.com/NVIDIA/NeMo-Agent-Toolkit ·
> 공식 문서 — https://docs.nvidia.com/nemo/agent-toolkit/

## 2. 아키텍처 & 핵심 기능

NAT 는 기존 에이전트 프레임워크를 **감싸는 계측 계층**으로 동작합니다. 크게 네 축입니다.

| 축 | 내용 |
|---|---|
| **Building (구축)** | 프레임워크 무관 에이전트 구성, 내장 UI, 재사용·커스터마이즈 |
| **Insights (관측)** | 프로파일링·관측성 — 워크플로를 **토큰 수준**까지 분석 |
| **Optimization (최적화)** | 평가 시스템, 프롬프트·하이퍼파라미터 최적화, RL 파인튜닝, 성능 프리미티브 |
| **Protocol (프로토콜)** | MCP · A2A 통합 |

```
[ 기존 프레임워크: LangChain / LlamaIndex / CrewAI / Semantic Kernel / Google ADK / custom ]
                         │  (NAT 가 감싸서 계측)
                         ▼
[ NAT: 워크플로 정의(workflow.yml) · 프로파일링 · 평가 · 관측 · MCP/A2A ]
                         │
                         ▼
              [ nat CLI: 실행(run) / 서비스화(serve) / 평가(eval) ]
```

- 워크플로는 **`workflow.yml`** 설정 파일로 **선언**하고, **`nat` CLI** 로 실행/평가합니다 — **코드를 쓰지 않습니다**.
- 특정 프레임워크에 종속되지 않으므로, 팀이 서로 다른 프레임워크를 써도 **공통 관측/평가 파이프라인**을 공유할 수 있습니다.

---

## 3. NAT 전용 환경 준비 (최초 1회)

아래 셀은 **NAT 전용 venv 가 없으면 만들고 `nvidia-nat` 을 설치**합니다. 이미 있으면 설치를 건너뜁니다.
CMD 로 직접 하면 다음과 같습니다.

```bat
REM 프로젝트 루트에서 실행
REM NAT 전용 가상환경 (강의 venv 와 분리 — 의존성 격리 목적)
uv venv notebooks\NAT_Tutorial\.venv-nat --python 3.11

REM 코어 + LangChain 플러그인 설치 (패키지명: nvidia-nat)
uv pip install --python notebooks\NAT_Tutorial\.venv-nat "nvidia-nat[langchain]"

REM 설치 확인
notebooks\NAT_Tutorial\.venv-nat\Scripts\nat.exe --version
```

> ⏱️ 최초 설치는 의존성이 많아 **수 분** 걸립니다. 이미 `NAT_Tutorial/` 시리즈 노트북용으로 설치해 두었다면 즉시 통과합니다.
>
> ⚠️ **venv 는 폴더를 옮기면 깨집니다** — 실행 파일에 절대 경로가 박혀 있어 이동 후에는
> `uv trampoline failed to canonicalize script path` 같은 오류가 납니다. 폴더를 옮겼다면 `.venv-nat` 을
> 지우고 위 명령으로 다시 만드세요.

In [2]:
# [설치] NAT 전용 venv 준비 — 없으면 생성 + nvidia-nat 설치, 있으면 건너뛴다
if os.path.exists(NAT_EXE):
    print(f"NAT 전용 venv 이미 준비됨 → {NAT_EXE}")
else:
    print("NAT 전용 venv 가 없습니다. 생성 + 설치를 시작합니다 (수 분 소요)")
    # 1) 전용 가상환경 생성 (강의 .venv 와 분리)
    utils.run_cmd(f'uv venv "{NAT_VENV}" --python 3.11')
    # 2) 코어 + LangChain 플러그인 설치 (extras 로 프레임워크 플러그인을 선택 설치)
    utils.run_cmd(f'uv pip install --python "{NAT_PY}" "nvidia-nat[langchain]"')

# 설치 결과 확인 — 여기서 버전이 찍히면 이후 셀을 실행할 수 있다
rc, out = utils.run_cmd(f'"{NAT_EXE}" --version', echo=False)
print("nat --version →", out.strip() if rc == 0 else f"실패(rc={rc}): {out.strip()[:200]}")

NAT 전용 venv 이미 준비됨 → c:\Users\stshin\Documents\GitHub\Agentic AI Tutorial\notebooks\NAT_Tutorial\.venv-nat\Scripts\nat.exe
nat --version → nat, version 1.8.0


---

## 4. 워크플로 선언 — `workflow.yml` 생성

NAT 워크플로는 **YAML 선언**입니다. 핵심 블록은 두 개입니다.

| 블록 | 역할 |
|---|---|
| `llms:` | 사용할 LLM 정의(이름 → 타입/모델/엔드포인트). `_type: openai` 는 **OpenAI 호환 엔드포인트**용 클라이언트 |
| `workflow:` | 워크플로 본체. 여기서는 가장 단순한 `chat_completion` 타입을 쓰고 위에서 정의한 llm 을 참조 |

본 강의의 모든 LLM 공급자(Ollama · llama.cpp · vLLM · NVIDIA build · OpenAI · OpenRouter)는
**OpenAI 호환 `/v1`** 을 제공하므로, `.env` 의 `LLM_PROVIDER` 값에서 `model_name` 과 `base_url` 만 뽑아
**같은 `_type: openai` 블록**으로 처리할 수 있습니다. 아래 셀이 그 변환을 수행합니다.

> 🔐 **API 키는 YAML 에 쓰지 않습니다.** 키를 파일에 적으면 노트북 출력·git 에 남을 수 있어,
> 표준 환경변수 `OPENAI_API_KEY` 로만 전달합니다(자식 프로세스인 `nat` CLI 가 상속).

In [3]:
# [설정] .env 의 LLM_PROVIDER → NAT llm 블록으로 변환해 workflow.yml 을 생성한다
def build_nat_llm_config(provider: str) -> dict:
    """공급자 이름을 NAT `_type: openai` 블록에 필요한 값으로 변환한다.

    본 강의의 공급자는 모두 OpenAI 호환 `/v1` 엔드포인트를 제공하므로,
    (모델명, base_url, api_key) 세 가지만 공급자별로 골라주면 된다.

    Args:
        provider: `.env` 의 LLM_PROVIDER 값.

    Returns:
        {"model_name": str, "base_url": str | None, "api_key": str} 형태의 딕셔너리.

    Raises:
        ValueError: OpenAI 호환 엔드포인트가 없는 공급자인 경우.
    """
    if provider == 'ollama':
        return dict(model_name=utils.OLLAMA_MODEL,
                    base_url=utils.OLLAMA_BASE_URL, api_key='ollama')
    if provider == 'llamacpp':
        return dict(model_name=utils.LLAMACPP_MODEL,
                    base_url=utils.LLAMACPP_BASE_URL, api_key='sk-no-key-required')
    if provider == 'vllm':
        return dict(model_name=utils.VLLM_MODEL,
                    base_url=utils.VLLM_BASE_URL, api_key='EMPTY')
    if provider == 'nvidia':
        # build.nvidia.com 의 OpenAI 호환 엔드포인트
        return dict(model_name=utils.NVIDIA_MODEL,
                    base_url=utils.NVIDIA_BASE_URL or 'https://integrate.api.nvidia.com/v1',
                    api_key=utils.NVIDIA_API_KEY)
    if provider == 'openrouter':
        return dict(model_name=utils.OPENROUTER_MODEL,
                    base_url=utils.OPENROUTER_BASE_URL, api_key=utils.OPENROUTER_API_KEY)
    if provider == 'openai':
        return dict(model_name='gpt-4o-mini', base_url=None, api_key=utils.OPENAI_API_KEY)
    if provider == 'google':
        # Gemini 의 OpenAI 호환 엔드포인트
        return dict(model_name='gemini-3.1-flash-lite',
                    base_url='https://generativelanguage.googleapis.com/v1beta/openai/',
                    api_key=utils.GOOGLE_API_KEY)
    raise ValueError(
        f"'{provider}' 는 OpenAI 호환 엔드포인트가 없어 이 워크플로에서 쓸 수 없습니다. "
        "notebooks/.env 의 LLM_PROVIDER 를 ollama/nvidia/google/openai/openrouter 중 하나로 바꾸세요.")


llm_cfg = build_nat_llm_config(utils.LLM_PROVIDER)

# base_url 이 없는 공급자(openai 기본 엔드포인트)는 해당 줄을 생략한다
base_url_line = f"    base_url: {llm_cfg['base_url']}\n" if llm_cfg['base_url'] else ""

workflow_yaml = f"""# 이 파일은 M03_4 노트북이 .env 의 LLM_PROVIDER({utils.LLM_PROVIDER}) 를 보고 생성했습니다.
# API 키는 보안을 위해 파일에 쓰지 않고 환경변수 OPENAI_API_KEY 로 전달합니다.
llms:
  course_llm:
    _type: openai                     # OpenAI 호환 클라이언트 (로컬/클라우드 공통)
    model_name: {llm_cfg['model_name']}
{base_url_line}    temperature: 0.0
    max_tokens: 512

workflow:
  _type: chat_completion              # 가장 단순한 워크플로 타입 (도구 없이 LLM 응답만)
  llm_name: course_llm
  system_prompt: |
    당신은 에이전트 시스템을 설명하는 한국어 어시스턴트입니다. 정확하고 간결하게 답하세요.
"""

with open(WORKFLOW_YML, 'w', encoding='utf-8') as f:
    f.write(workflow_yaml)

print(f"생성됨: {WORKFLOW_YML}\n" + "-" * 60)
print(workflow_yaml)

생성됨: c:\Users\stshin\Documents\GitHub\Agentic AI Tutorial\notebooks\nat_config\workflow.yml
------------------------------------------------------------
# 이 파일은 M03_4 노트북이 .env 의 LLM_PROVIDER(nvidia) 를 보고 생성했습니다.
# API 키는 보안을 위해 파일에 쓰지 않고 환경변수 OPENAI_API_KEY 로 전달합니다.
llms:
  course_llm:
    _type: openai                     # OpenAI 호환 클라이언트 (로컬/클라우드 공통)
    model_name: meta/llama-3.1-8b-instruct
    base_url: https://integrate.api.nvidia.com/v1
    temperature: 0.0
    max_tokens: 512

workflow:
  _type: chat_completion              # 가장 단순한 워크플로 타입 (도구 없이 LLM 응답만)
  llm_name: course_llm
  system_prompt: |
    당신은 에이전트 시스템을 설명하는 한국어 어시스턴트입니다. 정확하고 간결하게 답하세요.



### 4-1. `_type` 에 들어갈 수 있는 값

NAT 설정의 모든 블록은 **`_type` 으로 컴포넌트를 지정**합니다. 위 YAML 에서 쓴 `openai`(LLM)와
`chat_completion`(워크플로)도 그중 하나일 뿐입니다. 블록마다 **고를 수 있는 `_type` 목록이 다르며**,
그 목록은 **설치된 패키지(extras)가 결정**합니다 — 아래 값들은 이 노트북 환경(`nvidia-nat[langchain]` 1.8) 기준입니다.

#### 블록 ↔ 컴포넌트 종류

| YAML 블록 | 조회 명령 (`nat info components -t …`) | 대표 `_type` |
|---|---|---|
| `workflow:` · `functions:` | `function` | `chat_completion`, `react_agent`, `current_datetime` |
| `function_groups:` | `function_group` | `github` |
| `llms:` | `llm_provider` | `openai`, `nim`, `aws_bedrock`, `azure_openai`, `litellm`, `dynamo`, `oci`, `huggingface`, `huggingface_inference` |
| `embedders:` | `embedder_provider` | `openai`, `nim`, `azure_openai`, `huggingface` |
| `retrievers:` | `retriever_provider` | `milvus_retriever`, `nemo_retriever` |
| `memory:` | `memory` | (기본 설치엔 없음 — `nvidia-nat[mem0ai]`·`[redis]`·`[zep-cloud]` 등 extras 필요) |
| `object_stores:` | `object_store` | `in_memory` (extras 로 `s3`·`mysql`·`redis` 추가) |
| `middleware:` | `middleware` | `logging_middleware`, `cache`, `timeout`, `dynamic_middleware` |
| `authentication:` | `auth_provider` | `api_key`, `http_basic`, `oauth2_auth_code_flow` |
| `general.telemetry.tracing:` | `tracing` | `file`, `otelcollector`, `langsmith`, `langfuse`, `patronus`, `galileo`, `arize_ax` |
| `general.telemetry.logging:` | `logging` | `console`, `file` |
| `general.front_end:` | `front_end` | `console`(= `nat run`), `fastapi`(= `nat serve`) |
| `eval.evaluators:` | `evaluator` | `trajectory`, `tunable_rag_evaluator`, `langsmith`, `langsmith_judge`, `langsmith_custom` |
| `ttc_strategies:` | `ttc_strategy` | `best_of_n_selection`, `llm_as_a_judge_editor`, `multi_llm_generation` 등(실험적) |

#### `workflow:` 에 쓰는 `_type` — 에이전트·실행 패턴

`workflow:` 의 `_type` 은 곧 **"이 워크플로가 어떤 방식으로 동작하는가"** 입니다. 도구 없이 한 번 답할지,
ReAct 로 도구를 반복 호출할지, 여러 함수를 순차/병렬로 엮을지가 여기서 갈립니다.

| `_type` | 패키지 | 동작 |
|---|---|---|
| `chat_completion` | core | 도구 없이 LLM 응답만 — **이 노트북이 사용** |
| `react_agent` | langchain | **ReAct** — 사고(Thought)↔도구 호출(Action)↔관찰(Observation) 반복. 프롬프트 기반이라 도구 호출 미지원 모델도 가능 |
| `tool_calling_agent` | langchain | **네이티브 tool calling** 을 지원하는 모델용 에이전트 |
| `rewoo_agent` | langchain | **ReWOO** — 계획을 먼저 세우고 도구를 한 번에 실행(LLM 호출 절약) |
| `reasoning_agent` | langchain | 입력에 대해 추론한 결과를 다음 함수로 넘김 |
| `router_agent` | langchain | 입력을 여러 분기(branch) 중 하나로 라우팅 |
| `responses_api_agent` | langchain | OpenAI **Responses API** 기반 에이전트 |
| `per_user_react_agent` | langchain | 사용자별 상태를 유지하는 ReAct(멀티 유저 서비스용) |
| `auto_memory_agent` | langchain | 다른 에이전트를 감싸 **대화 메모리 저장/회수를 자동화** |
| `sequential_executor` · `parallel_executor` | langchain | 함수 목록을 **순차 / 병렬** 실행 |
| `langgraph_wrapper` | langchain | 기존 **LangGraph 그래프를 NAT 함수로 감싸기**(프레임워크 무관 계층의 실제 사례) |

#### `functions:` 에 쓰는 `_type` — 기본 제공 도구

에이전트의 `tool_names:` 에 넣을 도구들도 같은 `function` 레지스트리에서 옵니다.

| `_type` | 용도 |
|---|---|
| `current_datetime` · `current_timezone` | 현재 시각 / 타임존 |
| `code_generation` · `code_execution` | 코드 생성 / 샌드박스 실행 |
| `wiki_search` · `exa_internet_search` | 위키피디아 / 웹 검색 (검색 API 키 필요) |
| `nat_retriever` · `milvus_document_search` · `nvidia_rag` | 벡터스토어·RAG 검색 |
| `add_memory` · `get_memory` · `delete_memory` | 메모리 플랫폼 읽기/쓰기 |
| `github_files_tool` · `current_request_attributes` | GitHub 파일 조회 / 요청 속성 조회 |

> ⚠️ **버전·설치에 따라 달라집니다.** 위 목록은 외워야 할 상수가 아니라, **6-1절의 `nat info components` 로
> 언제든 다시 뽑을 수 있는 값**입니다. `nvidia-nat[llama-index]`·`[crewai]`·`[semantic-kernel]`·`[mcp]` 등
> 다른 extras 를 설치하면 그만큼 선택지가 늘어납니다.
>
> 예를 들어 `chat_completion` 대신 ReAct 에이전트로 바꾸려면 YAML 이 이렇게 됩니다(도구 하나 연결).
>
> ```yaml
> functions:
>   current_datetime:
>     _type: current_datetime
>
> workflow:
>   _type: react_agent
>   llm_name: course_llm
>   tool_names: [current_datetime]
> ```

---

## 5. 실행 — `nat run`

이제 **코드 한 줄 없이 YAML 만으로 정의한 워크플로**를 CLI 로 실행합니다.

```bat
REM 강의 커널이 아래 명령을 대신 실행합니다
notebooks\NAT_Tutorial\.venv-nat\Scripts\nat.exe run --config_file notebooks\nat_config\workflow.yml --input "질문"
```

출력에서 두 가지를 눈여겨보세요.

1. **Configuration Summary** — NAT 가 워크플로를 빌드하며 요약한 구성(Functions/LLMs/Memory/Retrievers 수).
   이것이 NAT 의 **관측(Insights)** 축이 동작하는 첫 지점입니다.
2. **Workflow Result** — 실제 LLM 응답.

In [4]:
# [실행] nat run — YAML 로 선언한 워크플로를 실제로 돌린다
# API 키는 명령줄에 노출되지 않도록 환경변수로만 전달한다(자식 프로세스가 상속).
os.environ['OPENAI_API_KEY'] = llm_cfg['api_key'] or ''

question = "에이전트 시스템에서 관측성(observability)이 왜 중요한지 두 문장으로 설명해줘."

rc, out = utils.run_cmd(
    f'"{NAT_EXE}" run --config_file "{WORKFLOW_YML}" --input "{question}"',
    echo=False,
)

if rc == 0:
    print(f"[성공] nat run (공급자={utils.LLM_PROVIDER})\n" + "=" * 60)
    print(out)
else:
    print(f"[실패] rc={rc} — 아래 로그를 확인하세요")
    print(out[-2000:])

[성공] nat run (공급자=nvidia)
2026-07-25 23:52:07 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'c:\Users\stshin\Documents\GitHub\Agentic AI Tutorial\notebooks\nat_config\workflow.yml'

Configuration Summary:
--------------------
Workflow Type: chat_completion
Number of Functions: 0
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2026-07-25 23:52:11 - INFO     - nat.runtime.session:329 - Shared workflow built (entry_function=None)
2026-07-25 23:52:12 - INFO     - httpx:1740 - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
C:\Users\stshin\Documents\GitHub\Agentic AI Tutorial\notebooks\NAT_Tutorial\.venv-nat\Lib\site-packages\nat\builder\function.py:380: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., me

---

## 6. 관측 (Observability)

NAT 의 **Insights(관측)** 축을 세 단계로 확인합니다.

| 단계 | 내용 | 질문 |
|---|---|---|
| 6-1 | 등록된 컴포넌트 조회 (`nat info components`) | 무엇이 **로드**되어 있는가? |
| 6-2 | LLM 송수신 관측 (`logging_middleware`) | LLM 에 **무엇이 실제로 전송**되었는가? |
| 6-3 | 검색 에이전트 관측 (`tool_calling_agent` + `wiki_search`) | 복잡한 워크플로는 **몇 번, 무엇을** 주고받는가? |

### 6-1. 등록된 컴포넌트 조회 — `nat info components`

NAT 는 **플러그인 레지스트리**를 갖고 있어, 설치된 패키지가 제공하는 컴포넌트(LLM 공급자, 도구, 워크플로 타입 등)를
CLI 로 조회할 수 있습니다. 어떤 확장이 실제로 로드되어 있는지 확인하는 **거버넌스 관점의 점검 도구**이자,
**4-1절의 `_type` 목록을 지금 이 환경 기준으로 다시 뽑는 방법**입니다.

```bat
REM 콘솔 표로 보기
notebooks\NAT_Tutorial\.venv-nat\Scripts\nat.exe info components -t llm_provider

REM 파일(JSON)로 저장해서 프로그램으로 다루기
notebooks\NAT_Tutorial\.venv-nat\Scripts\nat.exe info components -t llm_provider -o components.json
```

콘솔 표는 설명까지 포함해 매우 길기 때문에, 아래 셀은 **`-o` 옵션으로 JSON 을 받아** 종류별로 이름만 추려 보여줍니다.
출력의 각 묶음이 곧 **해당 YAML 블록에 쓸 수 있는 `_type` 후보 목록**입니다.

> 출력은 **패키지별로 묶여** 나옵니다. `nvidia-nat-*` 는 NAT 가 기본 제공하는 컴포넌트이고,
> 그 외 이름(예: `climate_agent`)이 보인다면 `NAT_Tutorial` 시리즈에서 **직접 만들어 등록한 커스텀 도구**입니다.
> 즉 **내가 만든 함수도 같은 레지스트리에 `_type` 으로 올라간다**는 뜻입니다 —
> 등록 방법은 [`NAT_03`](NAT_Tutorial/NAT_03_tools.ipynb) 에서 다룹니다.

In [5]:
# [관측] NAT 에 등록된 컴포넌트 조회 — -o 로 JSON 을 받아 이름/패키지만 정리해 본다
import json
import textwrap

def list_nat_components(component_type: str) -> list:
    """`nat info components -t <type>` 결과를 JSON 으로 받아 (이름, 패키지) 목록을 반환한다.

    콘솔 표 출력은 설명까지 포함해 매우 길고 열이 접히므로, `-o` 옵션으로 JSON 파일에
    저장한 뒤 파싱한다.

    Args:
        component_type: 조회할 컴포넌트 종류(예: 'llm_provider', 'front_end').

    Returns:
        (component_name, package) 튜플 리스트.
    """
    out_path = os.path.join(NAT_CONFIG_DIR, f'components_{component_type}.json')
    rc, log = utils.run_cmd(
        f'"{NAT_EXE}" info components -t {component_type} -o "{out_path}"', echo=False)
    if rc != 0:
        print(f"[실패] rc={rc}: {log[-300:]}")
        return []
    with open(out_path, encoding='utf-8') as f:
        results = json.load(f).get('results', [])
    return [(r['component_name'], r['package']) for r in results]


# 4-1절 표의 '조회 명령' 열과 같은 종류들 — 각 YAML 블록의 _type 후보가 그대로 나온다
COMPONENT_TYPES = {
    'function': 'workflow: / functions: 의 _type (에이전트·도구)',
    'llm_provider': 'llms: 의 _type',
    'embedder_provider': 'embedders: 의 _type',
    'retriever_provider': 'retrievers: 의 _type',
    'middleware': 'middleware: 의 _type',
    'tracing': 'general.telemetry.tracing: 의 _type',
    'evaluator': 'eval.evaluators: 의 _type',
    'front_end': 'general.front_end: 의 _type',
}

for ctype, purpose in COMPONENT_TYPES.items():
    items = list_nat_components(ctype)
    # 이름만 모아 패키지별로 묶어 본다 (nvidia-nat-* 는 공식 패키지, 그 외는 직접 등록한 것)
    by_package = {}
    for name, package in items:
        by_package.setdefault(package, []).append(name)

    print(f"\n[{ctype}] {len(items)}개 — {purpose}")
    for package, names in by_package.items():
        print(textwrap.fill(', '.join(sorted(names)),
                            width=100, initial_indent=f"  ({package}) ", subsequent_indent=' ' * 4))


[function] 40개 — workflow: / functions: 의 _type (에이전트·도구)
  (climate_agent) calculator_agent, country_average, country_average_buggy, country_stats,
    list_countries, warmest_coldest_year
  (nvidia-nat-core) add_memory, chat_completion, code_execution, current_datetime,
    current_request_attributes, current_timezone, delete_memory, execute_score_select_function,
    get_memory, github_files_tool, milvus_document_search, multi_llm_judge_function, nat_retriever,
    nvidia_rag, plan_select_execute_function, ttc_tool_orchestration, ttc_tool_wrapper
  (nvidia-nat-langchain) auto_memory_agent, code_generation, exa_internet_search, langgraph_wrapper,
    parallel_executor, per_user_react_agent, prompt_init, prompt_recombiner, react_agent,
    reasoning_agent, responses_api_agent, rewoo_agent, router_agent, sequential_executor,
    tavily_internet_search, tool_calling_agent, wiki_search

[llm_provider] 9개 — llms: 의 _type
  (nvidia-nat-core) aws_bedrock, azure_openai, dynamo, huggingface,

### 6-2. LLM 송수신 관측 — `logging_middleware`

앞의 `nat run` 출력은 **최종 답변만** 보여줍니다. 하지만 거버넌스에서 정작 중요한 것은
**"LLM 에 무엇이 실제로 전송되었는가"** 입니다 — 시스템 프롬프트, 사용자 입력, (에이전트라면) 도구 설명과
중간 사고까지 모두 외부 모델로 나갑니다. 여기서 프롬프트 유출·개인정보(PII) 포함·토큰 비용이 결정됩니다.

NAT 는 이를 위해 **미들웨어(middleware)** 계층을 제공합니다. `logging_middleware` 는 워크플로 코드를
전혀 건드리지 않고 **LLM 호출을 가로채** 입력 메시지와 응답(토큰 사용량 포함)을 로그로 남깁니다.
YAML 에 블록 하나만 추가하면 됩니다 — 이것이 NAT 가 말하는 **"감싸서 계측한다"** 의 실제 모습입니다.

```yaml
middleware:
  llm_logger:
    _type: logging_middleware   # LLM 호출을 가로채 입력/출력을 기록
    register_llms: true         # llms: 에 정의된 모든 LLM 을 자동 등록
    log_level: INFO
```

아래 두 셀은 **4절에서 만든 `workflow.yml` 에 이 블록만 덧붙인** `workflow_observed.yml` 을 만들고,
실행 로그에서 실제 송수신 내용을 추려 보여줍니다.

> 💡 미들웨어는 LLM 뿐 아니라 retriever·memory·object store·워크플로 함수도 같은 방식으로 가로챌 수 있고
> (`register_retrievers`, `register_workflow_functions` 등), `cache`·`timeout` 같은 다른 미들웨어도
> 같은 자리에 선언합니다. 컴포넌트 목록은 `nat info components -t middleware` 로 확인하세요.

In [6]:
# [설정] 4절의 workflow.yml 에 관측 미들웨어 블록만 덧붙인 workflow_observed.yml 을 만든다
OBSERVED_YML = os.path.join(NAT_CONFIG_DIR, 'workflow_observed.yml')

middleware_block = """
middleware:
  llm_logger:
    _type: logging_middleware       # LLM 호출을 가로채 입력/출력을 로그로 남기는 관측 미들웨어
    register_llms: true             # llms: 에 정의된 모든 LLM 을 자동으로 감싼다
    log_level: INFO                 # 로그 레벨 (DEBUG/INFO/WARNING/ERROR)
"""

# 워크플로 본체는 그대로 두고 관측만 덧붙인다 — "코드 수정 없는 계측"
with open(WORKFLOW_YML, encoding='utf-8') as f:
    base_yaml = f.read()

with open(OBSERVED_YML, 'w', encoding='utf-8') as f:
    f.write(base_yaml.rstrip() + '\n' + middleware_block)

print(f"생성됨: {OBSERVED_YML}")
print("추가된 블록만 표시\n" + "-" * 60)
print(middleware_block.strip())

생성됨: c:\Users\stshin\Documents\GitHub\Agentic AI Tutorial\notebooks\nat_config\workflow_observed.yml
추가된 블록만 표시
------------------------------------------------------------
middleware:
  llm_logger:
    _type: logging_middleware       # LLM 호출을 가로채 입력/출력을 로그로 남기는 관측 미들웨어
    register_llms: true             # llms: 에 정의된 모든 LLM 을 자동으로 감싼다
    log_level: INFO                 # 로그 레벨 (DEBUG/INFO/WARNING/ERROR)


In [7]:
# [관측] nat run 을 다시 돌리고, 미들웨어가 남긴 'LLM 에 실제로 전송된 메시지' 를 추출한다
import re

# 미들웨어 로그는 LangChain 메시지 객체의 repr 로 남으므로 정규식으로 필요한 부분만 뽑는다.
# 워크플로 종류에 따라 LLM 호출 방식이 ainvoke(한 번에) 또는 astream(토큰 스트리밍)으로 달라진다.
CALL_RE = re.compile(r"Calling (?:ainvoke|astream) with args:")
MSG_RE = re.compile(r"(SystemMessage|HumanMessage|AIMessage|ToolMessage)\(content='(.*?)', [a-z_]+=")
RESP_RE = re.compile(r"returned: content='(.*?)' [a-z_]+=")
TOOL_RE = re.compile(r"'name': '([\w.-]+)', 'args': (\{.*?\})")
USAGE_RE = re.compile(r"'(input_tokens|output_tokens|total_tokens)': (\d+)")


def _unescape(text: str) -> str:
    """로그에 repr 로 박제된 이스케이프(`\\n`·`\\uXXXX` 등)를 사람이 읽는 형태로 되돌린다."""
    text = text.replace('\\\\', '\\')
    text = re.sub(r'\\u([0-9a-fA-F]{4})', lambda m: chr(int(m.group(1), 16)), text)
    return text.replace('\\n', '\n').replace('\\t', '\t').replace('\\"', '"').replace("\\'", "'")


def parse_llm_traffic(log: str) -> list:
    """`nat run` 로그에서 logging_middleware 가 남긴 LLM 송수신 기록을 추출한다.

    미들웨어는 LLM 호출 직전에 `Calling ainvoke/astream with args: ...`,
    직후에 `Function ... returned: ...` 를 남긴다. 스트리밍(astream)이면 응답이
    토큰 조각으로 여러 줄에 걸쳐 나오므로 한 호출로 다시 이어 붙인다.

    Args:
        log: `utils.run_cmd` 가 돌려준 nat run 전체 출력(표준출력+표준에러).

    Returns:
        [{'messages': [(역할, 내용), ...], 'response': str,
          'tool_calls': [(도구명, 인자), ...], 'usage': {토큰명: 개수}}, ...] (호출 순서대로)
    """
    calls = []
    for line in log.splitlines():
        if CALL_RE.search(line):                       # LLM 으로 나가는 요청
            messages = [(role, _unescape(body)) for role, body in MSG_RE.findall(line)]
            calls.append({'messages': messages, 'response': '', 'tool_calls': [], 'usage': {}})
        elif 'returned:' in line and calls:             # LLM 이 돌려준 응답(또는 토큰 조각)
            matched = RESP_RE.search(line)
            if matched:
                calls[-1]['response'] += _unescape(matched.group(1))
            # 에이전트라면 응답에 도구 호출 지시가 담긴다
            for name, args in TOOL_RE.findall(line.split('returned:', 1)[1]):
                if (name, args) not in calls[-1]['tool_calls']:
                    calls[-1]['tool_calls'].append((name, args))
            usage = {name: int(count) for name, count in USAGE_RE.findall(line)}
            if usage:                                   # 토큰 사용량은 마지막 조각에만 실린다
                calls[-1]['usage'] = usage
    return calls


def print_llm_traffic(calls: list, max_chars: int = 400) -> None:
    """parse_llm_traffic 결과를 사람이 읽기 좋게 출력한다(긴 본문은 잘라서 표시).

    Args:
        calls: parse_llm_traffic 이 돌려준 LLM 호출 목록.
        max_chars: 메시지 한 건당 표시할 최대 글자 수.
    """
    def show(label: str, text: str, empty_note: str = '(내용 없음)') -> None:
        text = text.strip()
        if len(text) > max_chars:
            text = text[:max_chars] + f" …(이하 {len(text) - max_chars}자 생략)"
        print(f"  [{label}]")
        for line in text.splitlines() or [empty_note]:
            print('      ' + line)

    print(f"관측된 LLM 호출: {len(calls)}건\n" + "=" * 70)
    for i, call in enumerate(calls, 1):
        print(f"\n■ LLM 호출 #{i} — 실제 전송된 메시지 {len(call['messages'])}개")
        for role, body in call['messages']:
            show(role, body, empty_note='(내용 없음 — 도구 호출 지시만 담긴 메시지)')
        for name, args in call['tool_calls']:
            print(f"  [도구 호출] {name}({args})")
        show('응답', call['response'], empty_note='(텍스트 없음 — 도구 호출만 반환)')
        if call['usage']:
            usage = call['usage']
            print(f"  [토큰] 입력 {usage.get('input_tokens')} + 출력 {usage.get('output_tokens')}"
                  f" = 합계 {usage.get('total_tokens')}")


observed_question = "관측성(observability)을 한 문장으로 정의해줘."

rc, log = utils.run_cmd(
    f'"{NAT_EXE}" run --config_file "{OBSERVED_YML}" --input "{observed_question}"',
    echo=False,
)

calls = parse_llm_traffic(log)
print(f"[nat run rc={rc}] ", end='')
print_llm_traffic(calls)

if not calls:
    print("[안내] 관측 로그가 비어 있습니다. 아래 원본 로그 끝부분을 확인하세요.\n" + log[-1500:])

[nat run rc=0] 관측된 LLM 호출: 1건

■ LLM 호출 #1 — 실제 전송된 메시지 2개
  [SystemMessage]
      당신은 에이전트 시스템을 설명하는 한국어 어시스턴트입니다. 정확하고 간결하게 답하세요.
  [HumanMessage]
      관측성(observability)을 한 문장으로 정의해줘.
  [응답]
      관측성(observability)은 시스템의 상태를 관찰하고 측정할 수 있는 정도를 말합니다. 즉, 시스템의 내부 상태를 외부에서 어떻게 알 수 있는지에 대한 가능성을 나타냅니다.
  [토큰] 입력 80 + 출력 49 = 합계 129


#### 관측 결과에서 확인할 것

- **SystemMessage** — YAML 의 `system_prompt` 가 그대로 LLM 에 전달되었습니다. 노트북 코드 어디에도
  이 문자열을 보내는 코드는 없습니다. **설정이 곧 프롬프트**이며, 그래서 설정 파일이 거버넌스 감사 대상입니다.
- **HumanMessage** — `--input` 으로 넣은 사용자 질문. 실제 서비스라면 여기에 **개인정보가 섞여 외부 모델로
  나갈 수 있는 지점**이며, M03_1 의 가드레일(입력 레일)이 지키는 자리이기도 합니다.
- **토큰 사용량** — 입력/출력 토큰이 호출마다 기록됩니다. 이것이 **비용·지연의 원천 데이터**이고,
  NAT 의 프로파일링·최적화(Optimization) 축이 사용하는 값입니다.

여기까지는 `chat_completion` 이라 **LLM 호출이 딱 1건**이었습니다. 다음은 **도구를 쓰는 에이전트**를
같은 방식으로 관측해, 호출이 여러 건으로 늘어나는 모습을 봅니다.

### 6-3. 복잡한 워크플로 관측 — 검색 도구를 쓰는 에이전트

이번에는 **위키백과를 검색하는 에이전트 워크플로**를 같은 미들웨어로 관측합니다.
4-1절의 `_type` 표에서 두 가지를 골라 조립하면 끝입니다 — 코드는 여전히 없습니다.

```yaml
functions:
  wiki_search:                  # 검색 도구 (nvidia-nat[langchain] 에 기본 포함)
    _type: wiki_search
    max_results: 1

workflow:
  _type: tool_calling_agent     # 도구를 직접 호출하는 에이전트
  llm_name: course_llm
  tool_names: [wiki_search]
```

에이전트는 한 질문을 처리하며 **LLM 을 여러 번 호출**합니다. 대략 이런 흐름입니다.

```
사용자 질문 ──▶ [LLM 호출 #1] "어떤 도구를 어떤 인자로 부를까?"  ──▶ wiki_search 실행
                                                                        │ (검색 결과 = ToolMessage)
                 [LLM 호출 #2] 질문 + 도구 호출 + 검색 결과 전부 재전송 ◀┘  ──▶ 최종 답변
```

관측하면 **`chat_completion` 과 결정적으로 다른 점**이 드러납니다.

1. **호출이 여러 건** — 도구를 한 번 쓸 때마다 LLM 왕복이 한 번씩 늘어납니다(비용·지연도 그만큼).
2. **대화가 누적 전송** — 호출 #2 에는 원 질문뿐 아니라 **직전 도구 호출과 검색 결과 전문**이 다시 담깁니다.
   입력 토큰이 급증하는 이유이자, **검색해 온 외부 문서가 모델로 흘러 들어가는 지점**입니다.
3. **도구 호출 지시** — 응답에 텍스트 대신 `wiki_search({"question": ...})` 같은 **구조화된 호출**이 실려 옵니다.
   "에이전트가 무엇을 검색했는가"가 여기서 그대로 보입니다.

> ⚠️ 이 셀은 **위키백과(외부 서비스)에 실제로 접속**하고, 에이전트라 LLM 을 2회 이상 호출하므로
> 앞 셀보다 오래 걸립니다(수십 초). 작은 모델은 검색 결과를 요약하지 않고 그대로 옮기기도 하는데,
> **여기서 볼 것은 답변 품질이 아니라 "무엇이 오갔는가"** 입니다.
>
> 💡 `tool_calling_agent` 는 모델의 **네이티브 도구 호출** 기능을 씁니다(기본 `ollama`/`qwen3:8b`,
> `nvidia`/`llama-3.1-8b-instruct` 모두 지원). 도구 호출을 지원하지 않는 모델이라면
> `_type` 을 `react_agent` 로 바꾸면 프롬프트 기반(Thought/Action/Observation)으로 동작합니다 —
> 그때는 도구 설명이 통째로 시스템 프롬프트에 들어가는 모습이 관측됩니다.

In [8]:
# [설정] 검색 도구(wiki_search) + 에이전트(tool_calling_agent) + 관측 미들웨어 워크플로를 만든다
AGENT_YML = os.path.join(NAT_CONFIG_DIR, 'workflow_agent_observed.yml')

# llms 블록은 4절과 동일한 값(.env 의 LLM_PROVIDER)에서 다시 만든다 — base_url_line 은 4절에서 계산됨
agent_yaml = f"""# 이 파일은 M03_4 노트북 6-3절이 생성했습니다 (공급자: {utils.LLM_PROVIDER}).
llms:
  course_llm:
    _type: openai
    model_name: {llm_cfg['model_name']}
{base_url_line}    temperature: 0.0
    max_tokens: 512

functions:
  wiki_search:                        # 위키백과 검색 도구 — nvidia-nat[langchain] 에 기본 포함
    _type: wiki_search
    max_results: 1                    # 문서 1건만 (프롬프트가 지나치게 커지는 것을 막는다)

middleware:
  llm_logger:
    _type: logging_middleware         # 6-2 와 같은 관측 미들웨어를 그대로 재사용
    register_llms: true
    log_level: INFO

workflow:
  _type: tool_calling_agent           # 모델의 네이티브 도구 호출을 쓰는 에이전트
  llm_name: course_llm
  tool_names: [wiki_search]           # 에이전트가 쓸 수 있는 도구 목록
  max_tool_calls: 2                   # 도구 호출 상한 (무한 반복 방지)
  verbose: false
"""

with open(AGENT_YML, 'w', encoding='utf-8') as f:
    f.write(agent_yaml)

print(f"생성됨: {AGENT_YML}\n" + "-" * 60)
print(agent_yaml)

생성됨: c:\Users\stshin\Documents\GitHub\Agentic AI Tutorial\notebooks\nat_config\workflow_agent_observed.yml
------------------------------------------------------------
# 이 파일은 M03_4 노트북 6-3절이 생성했습니다 (공급자: nvidia).
llms:
  course_llm:
    _type: openai
    model_name: meta/llama-3.1-8b-instruct
    base_url: https://integrate.api.nvidia.com/v1
    temperature: 0.0
    max_tokens: 512

functions:
  wiki_search:                        # 위키백과 검색 도구 — nvidia-nat[langchain] 에 기본 포함
    _type: wiki_search
    max_results: 1                    # 문서 1건만 (프롬프트가 지나치게 커지는 것을 막는다)

middleware:
  llm_logger:
    _type: logging_middleware         # 6-2 와 같은 관측 미들웨어를 그대로 재사용
    register_llms: true
    log_level: INFO

workflow:
  _type: tool_calling_agent           # 모델의 네이티브 도구 호출을 쓰는 에이전트
  llm_name: course_llm
  tool_names: [wiki_search]           # 에이전트가 쓸 수 있는 도구 목록
  max_tool_calls: 2                   # 도구 호출 상한 (무한 반복 방지)
  verbose: false



In [9]:
# [관측] 에이전트 워크플로 실행 — 6-2 의 파서를 그대로 재사용해 LLM 왕복을 모두 본다
# 위키백과 검색이 필요한 질문을 던진다(모델이 도구를 쓸 수밖에 없게).
agent_question = ("위키피디아를 통해 딥러닝의 Transformer 모델에 대해서 검색하고, "
                  "한국어로 그게 몇 년도에 소개되었는지 알려줘.")

rc, agent_log = utils.run_cmd(
    f'"{NAT_EXE}" run --config_file "{AGENT_YML}" --input "{agent_question}"',
    echo=False,
)

agent_calls = parse_llm_traffic(agent_log)
print(f"[nat run rc={rc}] ", end='')
print_llm_traffic(agent_calls, max_chars=300)

# chat_completion(6-2) 과 비교 — 호출 수와 입력 토큰이 얼마나 늘었는지
total_in = sum(c['usage'].get('input_tokens', 0) for c in agent_calls)
total_out = sum(c['usage'].get('output_tokens', 0) for c in agent_calls)
print("\n" + "=" * 70)
print(f"[비교] chat_completion: LLM 호출 {len(calls)}건 / "
      f"입력 {sum(c['usage'].get('input_tokens', 0) for c in calls)} 토큰")
print(f"[비교] 검색 에이전트    : LLM 호출 {len(agent_calls)}건 / 입력 {total_in} 토큰, 출력 {total_out} 토큰")
print("→ 도구 호출 1회가 LLM 왕복 1회와 '누적된 대화 전체의 재전송' 을 부른다는 것을 토큰 수로 확인할 수 있다.")

if rc != 0:
    print(f"\n[안내] rc={rc} — 에이전트가 도중에 실패했더라도 위 관측 기록은 유효합니다. 원본 로그 끝부분:")
    print(agent_log[-1200:])

[nat run rc=0] 관측된 LLM 호출: 2건

■ LLM 호출 #1 — 실제 전송된 메시지 1개
  [HumanMessage]
      위키피디아를 통해 딥러닝의 Transformer 모델에 대해서 검색하고, 한국어로 그게 몇 년도에 소개되었는지 알려줘.
  [도구 호출] wiki_search({'question': '딥러닝 Transformer 모델 소개'})
  [응답]
      (텍스트 없음 — 도구 호출만 반환)
  [토큰] 입력 295 + 출력 24 = 합계 319

■ LLM 호출 #2 — 실제 전송된 메시지 3개
  [HumanMessage]
      위키피디아를 통해 딥러닝의 Transformer 모델에 대해서 검색하고, 한국어로 그게 몇 년도에 소개되었는지 알려줘.
  [AIMessage]
      (내용 없음 — 도구 호출 지시만 담긴 메시지)
  [ToolMessage]
      (내용 없음 — 도구 호출 지시만 담긴 메시지)
  [응답]
      Transformer 모델은 2017년 Vaswani et al.에 의해 소개되었다.
  [토큰] 입력 331 + 출력 19 = 합계 350

[비교] chat_completion: LLM 호출 1건 / 입력 80 토큰
[비교] 검색 에이전트    : LLM 호출 2건 / 입력 626 토큰, 출력 43 토큰
→ 도구 호출 1회가 LLM 왕복 1회와 '누적된 대화 전체의 재전송' 을 부른다는 것을 토큰 수로 확인할 수 있다.


---

## 7. 정리

1. **NAT 는 프레임워크 대체가 아니라 계측·관측·평가·최적화 계층** — LangChain/LlamaIndex/CrewAI 등을 감쌉니다.
2. **패키지명은 `nvidia-nat`**, 플러그인은 extras(`nvidia-nat[langchain]`), CLI 는 `nat`.
3. 워크플로는 **YAML 선언**(`llms:` + `workflow:`)이고, **`nat run`** 으로 코드 없이 실행됩니다.
   각 블록의 `_type` 후보는 **`nat info components`** 로 언제든 조회합니다(4-1·6-1절).
4. **관측도 YAML 한 블록**입니다 — `middleware: logging_middleware` 를 얹으면 워크플로를 고치지 않고
   **LLM 에 실제로 전송된 메시지·응답·토큰 사용량**을 그대로 볼 수 있습니다(6-2절).
5. **에이전트는 LLM 을 여러 번 호출**합니다(6-3절). 도구 호출 1회 = LLM 왕복 1회 + **누적 대화 전체 재전송**이며,
   검색해 온 외부 문서까지 프롬프트에 실립니다 — 비용·지연·데이터 유출을 함께 봐야 하는 이유입니다.
6. Python 3.11 로 올라오면서 NAT 의 버전 요구(`>=3.11,<3.13`)는 강의 환경과 **호환**됩니다.
   그럼에도 **전용 venv** 를 쓰는 이유는 NAT 가 LangChain 등 의존성을 자체 범위로 고정하기 때문입니다(**의존성 격리**).
7. **거버넌스 관점**에서 NAT 는 가드레일(안전)·트레이싱(관측)을 잇는 **프레임워크 무관 관측/평가 표준** 후보입니다.

### 더 깊이 — `NAT_Tutorial/` 시리즈 (전용 커널 `NAT (uv 3.11)`)

이 노트북은 "설치 → YAML 선언 → 실행 → 관측" 까지의 **최소 경로**만 다뤘습니다. 커스텀 도구·트레이싱 백엔드·평가·배포는
`notebooks/NAT_Tutorial/` 폴더의 7개 노트북에서 이어집니다.

| 노트북 | 내용 |
|---|---|
| [NAT_01](NAT_Tutorial/NAT_01_overview.ipynb) | NAT 개요와 환경 구성 |
| [NAT_02](NAT_Tutorial/NAT_02_first_workflow.ipynb) | 첫 워크플로 · `nat run` / `nat serve` |
| [NAT_03](NAT_Tutorial/NAT_03_tools.ipynb) | 커스텀 도구(function) 등록 |
| [NAT_04](NAT_Tutorial/NAT_04_observability.ipynb) | 관측(Phoenix 연동) |
| [NAT_05](NAT_Tutorial/NAT_05_multi_agent_math.ipynb) | 멀티 에이전트 |
| [NAT_06](NAT_Tutorial/NAT_06_evaluation.ipynb) | 평가(`nat eval`) |
| [NAT_07](NAT_Tutorial/NAT_07_deploy_with_NAT_UI.ipynb) | NAT UI 로 배포 |

> 설치·환경 구성 상세는 [`NAT_Tutorial/INSTALL_NAT.md`](NAT_Tutorial/INSTALL_NAT.md) 참고.

### 참고 자료 (출처)

- GitHub: https://github.com/NVIDIA/NeMo-Agent-Toolkit
- 공식 문서: https://docs.nvidia.com/nemo/agent-toolkit/
- API 키 발급: https://build.nvidia.com

---

### 모듈(거버넌스) 전체 구성

- (1) [NeMo Guardrails](M03_1_nemo_guardrails.ipynb) · (2) [LangSmith 트레이싱](M03_2_tracing.ipynb) ·
  (3) [Self-Refine](M03_3_self_refine.ipynb) · (4) NeMo-Agent-Toolkit 실습